In [6]:
import os
import re
import tempfile
import glob
from git import Repo
import networkx as nx

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI

# ─────────── 1. リポジトリをクローン ───────────
REPO_URL = "https://github.com/RWTH-EBC/AixLib.git"
tmp_dir = tempfile.mkdtemp()
Repo.clone_from(REPO_URL, tmp_dir)
print(f"Cloned to {tmp_dir}")

# ─────────── 2. Modelica ファイルを読み込み ───────────
def load_mo_files(root_dir):
    files = glob.glob(os.path.join(root_dir, "**", "*.mo"), recursive=True)
    return [f for f in files if os.path.isfile(f)]

mo_paths = load_mo_files(tmp_dir)
print(f"Found {len(mo_paths)} .mo files")

# ─────────── 3. クラス定義と継承関係をパース ───────────
class_defs = {}   # { class_name: full_text }
inheritances = [] # [(subclass, baseclass)]

for path in mo_paths:
    text = open(path, encoding="utf-8", errors="ignore").read()
    # クラス定義を全部抜き出し（簡易版）
    for m in re.finditer(r'^\s*(model|class|block|record)\s+(\w+)', text, re.MULTILINE):
        cls_name = m.group(2)
        # クラス本体を end …; まで抜く（やや雑ですがサンプル）
        body_pat = re.compile(rf'{m.group(0)}.*?end\s+{cls_name}\s*;', re.S)
        body_m = body_pat.search(text)
        class_defs[cls_name] = body_m.group(0) if body_m else m.group(0)

        # 継承 (extends) を探す
        for ext in re.finditer(r'extends\s+([\w\.]+)', m.group(0) + text[m.end():m.end()+200], re.IGNORECASE):
            base = ext.group(1).split('.')[-1]
            inheritances.append((cls_name, base))

print(f"Parsed {len(class_defs)} classes, found {len(inheritances)} inheritances")

# ─────────── 4. NetworkX でグラフ構築 ───────────
G = nx.DiGraph()
for cls in class_defs:
    G.add_node(cls)
for sub, base in inheritances:
    if base in class_defs:
        G.add_edge(base, sub)  # 親 -> 子

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ─────────── 5. クラス定義を embedding＆FAISS に登録 ───────────
api_key = os.getenv("XAI_API_KEY")
embedder = OpenAIEmbeddings(openai_api_key=api_key)

#embedder = OpenAIEmbeddings()
texts = [class_defs[c] for c in class_defs]
metadatas = [{"class": c} for c in class_defs]

# OpenAI の埋め込みエンドポイントに対して一度に送りすぎたトークン量が、あなたの組織に割り当てられた “tokens per minute (TPM)” の上限を超えてしまう
#index = FAISS.from_texts(texts, embedder, metadatas=metadatas)

# FAISS.from_texts の引数に chunk_size を渡して、
# embed_documents のバッチあたりのドキュメント数を減らす
index = FAISS.from_texts(
    texts,
    embedder,
    metadatas=metadatas,
    chunk_size=200  # 例えば 200 件ずつ埋め込み
)

Cloned to /var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/tmpa_3fzsxs
Found 4122 .mo files
Parsed 2239 classes, found 2576 inheritances
Graph: 2239 nodes, 1553 edges


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: xai-jpoa************************************************************************PNJt. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
# ─────────── 6. GraphRAG 風 Retriever の定義 ───────────
class GraphRAGRetriever:
    def __init__(self, vector_index, graph, k_sim=5, hops=1):
        self.index = vector_index
        self.graph = graph
        self.k = k_sim
        self.hops = hops

    def get_context(self, query: str):
        # 1) ベクトル検索で top-k 類似クラスを取得
        results = self.index.similarity_search_with_score(query, k=self.k)
        seed_classes = [r[0].metadata["class"] for r in results]
        
        # 2) グラフで隣接クラスを拾う
        context_ids = set(seed_classes)
        for cls in seed_classes:
            # 親子ともに拾う
            for nbr in nx.ego_graph(self.graph, cls, radius=self.hops).nodes:
                context_ids.add(nbr)
        
        # 3) コンテキスト文字列をまとめて返却
        pieces = []
        for cls in context_ids:
            pieces.append(f"### {cls}\n{class_defs[cls]}\n")
        return "\n".join(pieces)

# ─────────── 7. 質問応答の関数 ───────────
llm = ChatOpenAI(model_name="gpt-4", temperature=0)
retriever = GraphRAGRetriever(index, G, k_sim=3, hops=1)

def answer_query(query: str):
    ctx = retriever.get_context(query)
    prompt = (
        "あなたはModelica プロジェクトの専門家です。\n"
        "以下に関連クラスの定義があります。\n\n"
        f"{ctx}\n\n"
        f"質問: {query}\n"
        "回答してください。"
    )
    resp = llm(prompt)
    return resp.content

# ─────────── 8. 実行例 ───────────
if __name__ == "__main__":
    q = "ヒートポンプの Calibration クラスはどのように継承構造になっていますか？"
    print(answer_query(q))

In [ ]:
# ─────────── 0. 事前準備 ───────────
# これまでに行った…
# ・G: 継承関係の NetworkX DiGraph
# ・index: FAISS インデックス
# ・class_defs: クラス定義辞書
# ・GraphRAGRetriever クラス定義
#   （前の回答のコードをそのまま）

# ─────────── 1. Retriever のインスタンス化 ───────────
retriever = GraphRAGRetriever(
    vector_index=index,  # 先に作成した FAISS インデックス
    graph=G,             # 先に作成した NetworkX グラフ
    k_sim=3,             # 類似度検索の上位何件を種として拾うか
    hops=1               # グラフの何ホップまで隣接ノードを拾うか
)

# ─────────── 2. Grok3 クライアントの初期化 ───────────
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)

# ─────────── 3. 質問関数 ───────────
def answer_query(query: str) -> str:
    # 1) Retriever でコンテキスト抽出
    ctx = retriever.get_context(query)

    # 2) メッセージ構築
    messages = [
        {
            "role": "system",
            "content": (
                "あなたはModelicaプロジェクトの専門家です。\n"
                "以下に関連クラスの定義があります。参照して回答してください。\n\n"
                + ctx
            ),
        },
        {"role": "user", "content": query},
    ]

    # 3) Grok-3 へリクエスト
    completion = client.chat.completions.create(
        model="x-ai/grok-3-beta",
        messages=messages,
        temperature=0,
    )
    return completion.choices[0].message.content

# ─────────── 4. 実行例 ───────────
if __name__ == "__main__":
    q = "ヒートポンプの Calibration クラスはどのように継承構造になっていますか？"
    print(answer_query(q))